# Week 5 — Which Rules Survive Fresh Evidence?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/05/Week_05_Rules_Fresh_Evidence.ipynb)

**Driving question:** Does a selected rule survive fresh data and a skeptical reading?

Last week we distinguished a co-purchase pattern from an insight worth investigating. A high confidence can reflect a popular consequent; lift supplies a baseline comparison, but a large lift can still rest on little evidence. Today we keep the original claim visible while asking what happens in a later period. The interesting outcome is a better-supported recommendation, including a recommendation to wait.

By the end, you should be able to separate discovery choices from validation evidence; freeze a catalog and rule shortlist; recompute support, confidence, consequent prevalence, and lift without hiding failed rules; recognize rare-item effects and redundant directions; and revise a claim after a controlled challenge. We reuse the idea of held-out evaluation from representation learning, without treating a basket-rule arrow as a trained causal model.

Opening prediction: two rules both have 100% discovery confidence. One is supported by two antecedent baskets, the other by forty. What do you need before recommending either? Write a prediction about what could change in a later period.

The provisional 180-minute core excludes breaks: opening/setup 10; Reading A 15; worked counts 15; Reading B 15; split and discovery demonstration 15; Investigation 1 40; Reading C 15; Investigation 2 40; synthesis 15. Reading windows include reflection. Choose one route in each investigation; the other is optional. Activities are formative; no assignment or deadline is introduced.

## Start with visual intuition

Before the detailed explanations, choose **one** short exploration below. Use 3–5 minutes **within the existing 10-minute opening/setup block**, then return to the notebook. The other link is optional further study. No account or upload is needed. If a site is unavailable, the notebook's own count tables and figures provide the core explanation.

1. **New intuition for this week — why a fraction needs enough observations:** [Seeing Theory — Point Estimation and the Bootstrap](https://seeing-theory.brown.edu/frequentist-inference/index.html). Begin in **Point Estimation**: click **Drop 100 Samples**, note the estimate, then **Drop 1000 Samples**. Notice how an estimate can change as observations accumulate; a larger sample does not guarantee that every update moves closer to the target. **Bring back one sentence:** why should 2/2 and 40/40 deserve different scrutiny despite the same fraction? The page estimates π from randomly dropped points, not association-rule lift. Its archived site remains interactive. For optional depth later, try **The Bootstrap**: choose Normal, click Sample, then Resample 100 times. This illustrates resampling a sample mean; it does not justify treating repeated customer invoices as independent.

2. **Quick bridge from Week 4 — the baseline behind confidence:** [Setosa — Conditional probability explained visually](https://setosa.io/conditional/). Here **P(B|A)** means the fraction with B among observations with A. Set P(A) to 0.2, P(B) to 0.2, and the overlap P(A ∩ B) to 0.1. Confidence is 0.5; dividing by B's baseline 0.2 gives lift 2.5. Change P(B) to 0.4, then reset the overlap to 0.1 if it moved. Confidence stays 0.5, but lift becomes 1.25. Switch between **P(B|A)** and **P(A|B)**. **Bring back one sentence:** how can unchanged confidence accompany a changed lift? The controls can adjust overlap together, so check all displayed values. The animation illustrates conditioning, not purchase order or a causal intervention.

The useful connection is **finite evidence → a conditional fraction → its baseline → a separate check**. Technical references appear at the start of the relevant readings, where you can use them as needed; they are not extra required reading before beginning.


## Setup — run this notebook independently

Connect to a CPU runtime in Colab and choose **Runtime → Run all**. Dependencies install automatically; the public course dataset downloads, passes a checksum check, and is cached. There is no manual upload, private account, or API key. If an installation requests a restart, restart and run all. The badge becomes usable after this lecture is published in the student repository.

Locally, run `uv sync --locked`, then `uv run jupyter lab` from the course root. The bundled retail extract is an offline fallback once dependencies are installed. No earlier notebook outputs or exports are required. The [data guide](../../data/README.md) records attribution and limitations. First-time installation/download needs internet; optional readings are not required to execute the notebook.

Run the defaults first. Before changing a route or setting, record your prediction. Rerun that cell and every dependent cell; restart and run all if state is unclear. Keep one saved default run so a changed result can be traced to an explicit choice. Running all reveals the complete demonstration; meaningful independent validation requires recording your choices before reading its later outputs.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
        'numpy>=2,<3','pandas>=2.2,<3','matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8','mlxtend==0.23.4'])
    print('Colab dependencies ready. Continue, or restart and run all if requested.')
else:
    print('Local runtime: using the locked course environment.')

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlxtend.frequent_patterns import fpgrowth, association_rules
import mlxtend

pd.set_option('display.max_columns', 14)
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| mlxtend:', mlxtend.__version__)

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## Reading A — Discovery produces a claim; validation challenges it

**Read independently for 10–15 minutes, including the checkpoint.** Try to explain why a rule can be calculated correctly yet still disappoint when checked again.

### Recall the measures before judging them

Use one visual warm-up from the beginning of the notebook, then return here. Let A be the **antecedent**, the item whose presence defines the group we inspect, and B the **consequent**, the co-occurring item. Among N baskets, let n_A contain A, n_B contain B, and n_AB contain both. Support is n_AB/N; confidence is n_AB/n_A; B's baseline prevalence is n_B/N; and lift is confidence divided by that baseline. Each proportion must use counts from the same period. A zero denominator can make a measure undefined, as the worked examples below show.

**Optional metric reference, when needed:** [mlxtend — Association rules](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/). Focus on support, confidence, and lift. Explain why counting a frozen rule is different from mining another qualifying rule list. The notebook supplies the definitions and calculations; extra metrics in the reference are not required vocabulary.

### Searching changes what a striking result means

Imagine inspecting one combination suggested before opening a dataset. Now imagine searching hundreds of combinations and reporting only the most impressive. Those are different evidence-generating procedures. Even when products are unrelated in the underlying process, some observed combinations will look unusually strong in a finite sample. Selecting the winner favors positive fluctuations. This is the multiple-discovery problem: the number and flexibility of the searches affect how surprising the selected result should seem.

This does not mean that every large lift is accidental. It means that the discovery score alone does not tell us how reliably the selected relationship will recur. Lowering a support threshold, trying several catalogs, reversing arrows, and changing date boundaries can all create more opportunities to find an attractive story. An honest description names those choices rather than presenting the final rule as the only question ever asked.

Recall the previous course's training and evaluation distinction. A model can fit quirks of the examples used to choose it. Here we may not train a predictive model, but we still choose products, thresholds, and rules from data. Those choices adapt to the observed sample. **Discovery data** are the observations used to make those choices. **Validation data** are separate observations used to examine the resulting, already specified claim. “Unsupervised” does not mean “no risk of overfitting.” We can overfit an interpretation as well as a neural network.

### Freeze a question that could actually fail

To freeze a shortlist is to record the exact rules and their selection procedure before inspecting validation results. Record the basket definition, eligible purchases, catalog, mining threshold, selection order, and rule direction. Do not replace a disappointing rule with a new validation winner and continue calling the new score held-out evidence. The disappointing result belongs in the table: it is part of what we learned.

The distinction is easiest to see with a deliberately constructed example. In ten discovery baskets, A occurs twice, B twice, and they always occur together. The joint count is two. Confidence for A → B is 2/2 = 1 and B's baseline prevalence is 2/10 = 0.2, so lift is five. That is a correct description of these ten baskets. It is also a very small evidence base: changing one of those two A baskets from “both” to “A only” would halve confidence.

Now keep the rule fixed and inspect twenty new baskets. A occurs four times, B ten times, and both occur twice. Confidence is 2/4 = 0.5, while B's baseline is 10/20 = 0.5. Lift is now one. The joint count stayed two, yet both the conditional fraction and the baseline changed. The rule did not contain a coding error; the new observations simply do not reproduce the original excess co-occurrence. Do not explain the change by comparing joint counts alone.

### A failed check must remain visible

If validation contains A and B but never together, confidence and lift are zero. If A is absent, confidence is undefined because its denominator is zero. If B is absent, lift is undefined because the baseline denominator is zero; confidence can still be zero when A occurs. An empty validation population leaves all proportions undefined. These are different situations, so the notebook preserves counts and explanatory status labels alongside missing numerical values.

A tempting shortcut is to mine each partition and join the surviving rule lists. That hides exactly the cases we most need to see. A rule with no validation co-occurrence will not be mined there. A rule with a few occurrences may fail the mining threshold even though its metrics can be calculated. Instead, evaluate every frozen rule directly from the validation baskets. Mining is a discovery procedure; counting a specified rule is an evaluation procedure.

### What a successful check would mean

Persistence in a later period supports a claim about recurrence under the stated conditions. It does not prove that recommending one product causes sales of another. Buyers may have shared purposes, see the same promotion, or choose complementary products without any recommendation. Likewise, a weaker later association can reflect seasonality, availability, customer mix, sampling variation, or an originally fragile finding. Our counts help narrow the explanation; they do not identify its cause by themselves.

### Checkpoint

Calculate support, confidence, baseline, and lift in both toy periods. Explain why equal joint counts do not imply equal evidence. What would an inner join of independently mined rule lists hide? Write one way to make validation fail as a procedure even when the arithmetic is correct.

<details><summary>Self-check — open after writing</summary>

Discovery: support 2/10 = 0.2, confidence 1, baseline 0.2, lift 5. Validation: support 2/20 = 0.1, confidence 0.5, baseline 0.5, lift 1. Both the antecedent and population denominators matter. An inner join hides rules absent from the second mined list. Choosing replacements after seeing validation, or fitting the catalog on both periods, lets validation information influence selection.

</details>

In [ ]:
def rule_evidence(presence, antecedent, consequent):
    A, B = frozenset(antecedent), frozenset(consequent)
    if not A or not B or A & B:
        raise ValueError('Use nonempty disjoint antecedent and consequent sets.')
    if not presence.dtypes.eq(bool).all():
        raise ValueError('Use a boolean presence table, not quantities or missing values.')
    def contains(items):
        if not items.issubset(presence.columns):
            return pd.Series(False,index=presence.index)
        return presence[sorted(items)].all(axis=1)
    in_A, in_B = contains(A), contains(B)
    N, n_A, n_B = len(presence),int(in_A.sum()),int(in_B.sum())
    n_AB = int((in_A & in_B).sum())
    support = n_AB/N if N else np.nan
    confidence = n_AB/n_A if n_A else np.nan
    baseline = n_B/N if N else np.nan
    lift = confidence/baseline if n_A and n_B and N else np.nan
    return pd.Series({'baskets':N,'A_count':n_A,'B_count':n_B,'joint_count':n_AB,
        'support':support,'confidence':confidence,'B_baseline':baseline,'lift':lift})


def constructed_baskets(both, A_only, B_only, neither):
    # Four mutually exclusive cells; intentionally synthetic, not retail observations.
    rows = ([[True, True]] * both + [[True, False]] * A_only
            + [[False, True]] * B_only + [[False, False]] * neither)
    return pd.DataFrame(rows, columns=['A', 'B'], dtype=bool)

toy_discovery = constructed_baskets(2, 0, 0, 8)
toy_validation = constructed_baskets(2, 2, 8, 8)
toy_comparison = pd.DataFrame({
    'discovery': rule_evidence(toy_discovery, {'A'}, {'B'}),
    'validation': rule_evidence(toy_validation, {'A'}, {'B'})})
display(toy_comparison.round(3))

# TODO 1: Before changing an observation, predict the change in confidence and lift.
remove_one_success = False
toy_changed = toy_discovery.copy()
if remove_one_success:
    toy_changed.loc[0, 'B'] = False  # Keep the basket, change one B presence.
display(rule_evidence(toy_changed, {'A'}, {'B'}).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, metric, limit in zip(axes, ['confidence', 'lift'], [1.1, 5.5]):
    values = toy_comparison.loc[metric]
    ax.bar(values.index, values.to_numpy(float), color=['#476f95', '#d58b47'])
    ax.set(title=f'Frozen synthetic A → B: {metric}', ylabel=metric.capitalize(), ylim=(0, limit))
    for i, value in enumerate(values): ax.text(i, value + limit * .02, f'{value:g}', ha='center')
axes[1].axhline(1, color='black', linestyle='--', linewidth=1, label='Marginal baseline (lift = 1)')
axes[1].legend(fontsize=9)
fig.suptitle('Same joint count (2), different denominators — no uncertainty intervals implied')
plt.tight_layout(); plt.show()
# TODO 2: Explain why neither panel can be interpreted without the count table.

## Reading B — Keep the comparison honest from the first line

**Read and reflect for 10–15 minutes.** Identify which choices must be fixed before validation, and which quantities must be recalculated separately in each period.

### See the comparison before building it

| Observations | What we do with them |
|---|---|
| Earlier invoices: discovery | Choose the catalog and threshold → mine → rank → freeze exact directed rules |
| Later invoices: validation | Use the frozen catalog and rules → count all outcomes → qualify the claim |

The second row does not send a preferred result back to the first row. Revising after seeing it is a new exploratory choice, not the same held-out check.

**Optional reference before the implementation:** [scikit-learn 1.7 — Data leakage](https://scikit-learn.org/1.7/common_pitfalls.html#data-leakage), matching our local library series. Read the explanation of preprocessing fitted only on training data and connect it to our discovery-only catalog. Its examples use predictive estimators; we apply the same separation to pattern selection, not supervised accuracy. The table and prose here are sufficient to continue without opening the documentation.

### A date split answers a particular question

Our case uses the same attributed UCI Online Retail teaching extract as earlier lectures, under Creative Commons Attribution 4.0. It contains selected customer histories rather than a representative sample of the retailer. We keep identified, positive, noncancelled product purchases. Stock codes matching five digits and an optional capital letter are our transparent goods-code heuristic. This excludes many adjustments but is not a verified product classification. We remove exact duplicate lines, preserve raw data, and interpret an invoice as one binary product-presence basket. We are studying positive purchases, not reconstructing net revenue after returns.

For this exercise, invoices dated before 1 June 2011 form discovery; invoices on or after that date form validation. The cutoff is a stated teaching design chosen for usable date coverage, not optimized to make a particular rule succeed. Each invoice has one date and one identified customer in these eligible records; the code checks this before splitting. Lines from the same invoice must stay together. Otherwise the same purchase event could appear on both sides and create a misleading impression of fresh evidence.

The chronological split asks whether an early-period association recurs in later purchases from this selected extract. It does not ask whether we can generalize to entirely new customers. The same customer can contribute invoices to both periods, and multiple invoices from that customer may be dependent. A customer-disjoint split would answer a different question. We report overlap and distinct customer counts so the evaluation unit is visible. We do not use invoice counts as a claim of independently sampled shoppers.

There is another limitation to name explicitly: Week 4 already explored this full extract. The later partition is held out from today's discovery code, but it is not historically unseen during the course. This is a retrospective demonstration of the validation workflow, not a pristine prospective test of last week's findings. A genuinely new final claim would need evidence not previously used to develop it.

### Learn the catalog on discovery only

We rank stock codes by distinct discovery-invoice occurrence, break ties by code, and retain forty products. Then we use exactly those columns in both periods, including products absent later. Labels displayed during selection also come from discovery. A product that becomes common only in validation is outside this discovery catalog. That is a coverage limitation to report, not a reason to refit the vocabulary after seeing the answer.

Every eligible invoice remains in its period's basket table, even when it contains none of the forty products. An all-false row means “no product from our chosen catalog,” not “no purchases.” Dropping these rows changes the population used to calculate product prevalence, support, and lift. Do not silently condition the later period on buying something from the catalog. Similarly, count repeated product lines or quantities as one presence per invoice, not extra independent support.

The search uses frequent pattern growth (FP-growth), as in Week 4. We bound itemsets to length two and require eight joint discovery occurrences. This produces singleton-to-singleton rules in both directions. The lower count boundary than Week 4 reflects the smaller discovery partition; it is an explicit teaching choice, not proof that eight observations make a finding reliable. Candidate count, selected catalog, and cutoff are part of the result's scope. If you change them, you have changed the discovery procedure.

### Three distinct claims are better than three copies

Reversing A → B changes confidence, but joint support and lift remain the same for that pair within a fixed population. Reporting both arrows as two independently confirmed relationships would exaggerate the evidence. Our shortlist therefore keeps one direction per unordered product pair, choosing direction by the selected discovery ranking. The practical question still depends on that direction: which conditional purchase fraction would you discuss with a colleague?

Redundancy can be broader than reversed pairs. Several rules may describe variants of one product family, shared promotion, or packaging theme. A longer rule such as {A, C} → B might add little beyond A → B if C is nearly always present with A. Longer itemsets are outside today's bounded core, but the principle applies: more rows do not necessarily mean more distinct knowledge. Group related findings in your interpretation and ask what each additional rule contributes.

### Compare like quantities, not just attractive scores

For each frozen rule, record the basket count N, antecedent count, consequent count, and joint count in both periods. Recalculate each period's support, confidence, consequent baseline, and lift from its own counts. Do not divide validation confidence by discovery prevalence: that would mix populations. More validation baskets can yield more joint occurrences even when support decreases; a lower lift can coexist with a useful conditional fraction if the consequent became common. Describe the whole comparison.

The status “observed co-occurrence” below only means the joint count is positive. It is not a pass label, a significance test, or a business recommendation. A positive joint count with lift below one is still co-occurrence. Preserve undefined values, and keep zero-count findings in the table. No universal lift threshold turns a descriptive comparison into a validated intervention.

### Checkpoint

List four frozen choices and four quantities recalculated on validation. Why can the later basket table have all-false rows and absent columns? Why does reversing a rule not give another independent confirmation? Explain what generalization claim our recurring customers prevent.

<details><summary>Self-check</summary>

Freeze eligibility, invoice unit, catalog, mining threshold, selection order, and rule sides. Recalculate period counts, support, confidence, baseline, and lift. All-false rows preserve the invoice population; discovery products retain their columns even when absent later. Reverse directions share a joint event. Recurring customers mean this later-period check does not establish performance on unseen customers or independent customer samples.

</details>

In [ ]:
eligible = raw.drop_duplicates().copy()
eligible = eligible[(~eligible.InvoiceNo.str.upper().str.startswith('C', na=False)) &
    eligible.Quantity.gt(0) & eligible.UnitPrice.gt(0) & eligible.CustomerID.notna() &
    eligible.StockCode.str.fullmatch(r'\d{5}[A-Z]?', na=False)].copy()
invoice_consistency = eligible.groupby('InvoiceNo')[['InvoiceDate', 'CustomerID']].nunique()
assert invoice_consistency.eq(1).all().all(), 'An invoice must have one date and customer.'
cutoff = pd.Timestamp('2011-06-01')
discovery_lines = eligible[eligible.InvoiceDate.lt(cutoff)].copy()
validation_lines = eligible[eligible.InvoiceDate.ge(cutoff)].copy()
if discovery_lines.empty or validation_lines.empty:
    raise ValueError('Both time periods need eligible invoices. Review the cutoff.')
discovery_ids = pd.Index(sorted(discovery_lines.InvoiceNo.unique()), name='InvoiceNo')
validation_ids = pd.Index(sorted(validation_lines.InvoiceNo.unique()), name='InvoiceNo')
assert discovery_ids.intersection(validation_ids).empty

def discovery_catalog(lines, size=40):
    counts = (lines[['InvoiceNo', 'StockCode']].drop_duplicates().StockCode.value_counts()
              .rename_axis('StockCode').reset_index(name='invoice_count')
              .sort_values(['invoice_count', 'StockCode'], ascending=[False, True]))
    return list(counts.head(size).StockCode)

catalog = discovery_catalog(discovery_lines)
description_counts = (discovery_lines.dropna(subset=['Description'])
    .groupby(['StockCode', 'Description']).size().rename('rows').reset_index()
    .sort_values(['StockCode', 'rows', 'Description'], ascending=[True, False, True]))
labels = description_counts.drop_duplicates('StockCode').set_index('StockCode').Description.to_dict()

In [ ]:
def make_presence(lines, unit, vocabulary, universe):
    if len(set(vocabulary)) != len(vocabulary) or not universe.is_unique:
        raise ValueError('Catalog and basket IDs must be unique.')
    pairs = lines.loc[lines.StockCode.isin(vocabulary),[unit,'StockCode']].drop_duplicates()
    if pairs.empty:
        return pd.DataFrame(False,index=universe,columns=vocabulary,dtype=bool)
    table = pd.crosstab(pairs[unit],pairs.StockCode)
    return table.reindex(index=universe,columns=vocabulary,fill_value=0).gt(0)


discovery = make_presence(discovery_lines, 'InvoiceNo', catalog, discovery_ids)
validation = make_presence(validation_lines, 'InvoiceNo', catalog, validation_ids)
assert discovery.columns.equals(validation.columns)
coverage = []
for name, lines, table in [('discovery', discovery_lines, discovery),
                           ('validation', validation_lines, validation)]:
    coverage.append({'period': name, 'first_date': lines.InvoiceDate.min(),
        'last_date': lines.InvoiceDate.max(), 'invoices': len(table),
        'customers': lines.CustomerID.nunique(), 'catalog_empty_invoices': int((~table.any(axis=1)).sum())})
display(pd.DataFrame(coverage).set_index('period'))
shared_customers = set(discovery_lines.CustomerID) & set(validation_lines.CustomerID)
print('Customers appearing in both periods:', len(shared_customers))
# TODO 3: Explain the evaluation unit and inspect one all-false invoice's source lines.
empty_ids = discovery.index[~discovery.any(axis=1)]
if len(empty_ids):
    display(discovery_lines[discovery_lines.InvoiceNo.eq(empty_ids[0])]
            [['InvoiceNo', 'StockCode', 'Description', 'Quantity']].head(5))

In [ ]:
RULE_COLUMNS = ['A','B','A_label','B_label','baskets','A_count','B_count',
                'joint_count','support','confidence','B_baseline','lift']

def mine_pair_rules(presence, min_joint_count=12):
    if not isinstance(min_joint_count,int) or isinstance(min_joint_count,bool) or min_joint_count<1:
        raise ValueError('Minimum joint count must be a positive integer.')
    if len(presence)==0 or len(presence.columns)<2 or min_joint_count>len(presence):
        return pd.DataFrame(columns=RULE_COLUMNS)
    # nextafter prevents floating rounding from excluding an exact boundary count.
    threshold = np.nextafter(min_joint_count/len(presence),0.0)
    itemsets = fpgrowth(presence,min_support=threshold,use_colnames=True,max_len=2)
    if itemsets.empty or not itemsets.itemsets.map(len).eq(2).any():
        return pd.DataFrame(columns=RULE_COLUMNS)
    generated = association_rules(itemsets,num_itemsets=len(presence),metric='confidence',min_threshold=0)
    rows = []
    for _,rule in generated.iterrows():
        A,B = next(iter(rule.antecedents)),next(iter(rule.consequents))
        rows.append({'A':A,'B':B,'A_label':labels.get(A,A),'B_label':labels.get(B,B),
            'baskets':len(presence),'A_count':int(round(rule['antecedent support']*len(presence))),
            'B_count':int(round(rule['consequent support']*len(presence))),
            'joint_count':int(round(rule.support*len(presence))),
            'support':rule.support,'confidence':rule.confidence,
            'B_baseline':rule['consequent support'],'lift':rule.lift})
    result = pd.DataFrame(rows,columns=RULE_COLUMNS)
    return result[result.joint_count.ge(min_joint_count)].sort_values(['A','B']).reset_index(drop=True)


minimum_count = 8
candidates = mine_pair_rules(discovery, minimum_count)
print('Discovery directed candidates:', len(candidates), '| Catalog:', len(catalog),
      '| Discovery baskets:', len(discovery), '| Joint count minimum:', minimum_count)
display(candidates[['A', 'B', 'joint_count', 'confidence', 'B_baseline', 'lift']].head(5).round(3))

## Investigation 1 — Freeze three claims, then check them

**40 minutes. Choose one route.** Both routes work from the same discovery candidates; neither is allowed to choose winners using later scores. Defaults are runnable. Work with the interpretation, not with manual data preparation.

**Route A — `lift`:** rank by discovery lift, breaking ties by joint count and codes. Keep three distinct unordered pairs. Predict which conditional fractions might change. Do not predict that every large lift must disappear: persistent patterns are possible too.

**Route B — `confidence`:** rank by discovery confidence, then joint count and codes. Keep three distinct unordered pairs. Explain what each rule adds beyond its consequent's discovery prevalence. Compare the number of antecedent baskets behind similar-looking conditional fractions.

1. Spend five minutes inspecting discovery counts and descriptions. Write your route choice and one sentence explaining each selected rule. Record the cutoff, catalog size, joint threshold, and ranking choice as your freeze record. This cell prints a digest of the exact rule identities for comparison later; a digest records identity, not the time at which you made a decision.
2. Before scrolling to evaluation, predict which counts or denominators could change. Take ten minutes to evaluate those exact rules on validation, including zeros or undefined measures. Do not remove an inconvenient row or replace it with another rule.
3. Spend ten minutes explaining all three comparisons in plain language. “Joint count rose” and “lift fell” are observations; explain which denominators make them compatible. A change in rank is not a measure of predictive accuracy.
4. Use ten minutes for source inspection. For one selected rule, inspect one matching validation invoice and one antecedent-without-consequent invoice when available. Inspect descriptions in their original context. If an example is absent, state that instead of inventing one. Reverse the arrow and explain the changed conditional question.
5. Finish with a five-minute peer explanation: what persisted, what weakened, and what evidence is still missing? A surviving association supports recurrence in this extract, not a causal promotion plan.

**Done when:** you have a recorded freeze, three rule comparisons with counts and baselines, an actual source inspection or a documented absence, and one revised claim. If your discovery settings yield fewer than three candidates, report the available number. An empty discovery list is an honest outcome, not a reason to search validation for replacements.

<details><summary>Hint if you are stuck</summary>

For the first rule, say: “Among [antecedent count] earlier A invoices, [joint count] also contained B; later it was [joint count] of [antecedent count]. B's prevalence changed from ... to ... .” Only then discuss lift. The separate period denominators are printed below.

</details>

In [ ]:
# TODO 4: Choose a discovery-only ranking BEFORE reading validation results.
selection_route = 'lift'  # alternative: 'confidence'

def freeze_shortlist(candidate_table, route='lift', size=3):
    if route not in {'lift', 'confidence'}:
        raise ValueError('Choose lift or confidence.')
    ranked = candidate_table.sort_values([route, 'joint_count', 'A', 'B'],
                                          ascending=[False, False, True, True]).copy()
    ranked['pair'] = [tuple(sorted((A, B))) for A, B in zip(ranked.A, ranked.B)]
    return ranked.drop_duplicates('pair').head(size)[['A', 'B']].copy().reset_index(drop=True)

frozen_rules = freeze_shortlist(candidates, selection_route)
freeze_record = {'cutoff': str(cutoff.date()), 'catalog_size': len(catalog),
    'minimum_joint_count': minimum_count, 'ranking': selection_route,
    'catalog': tuple(catalog), 'rules': tuple(map(tuple, frozen_rules.to_numpy()))}
freeze_digest = hashlib.sha256(repr(freeze_record).encode()).hexdigest()[:16]
print('Freeze identity:', freeze_digest)
print({k: v for k, v in freeze_record.items() if k not in {'catalog', 'rules'}})
display(frozen_rules.assign(A_label=frozen_rules.A.map(labels), B_label=frozen_rules.B.map(labels)))
# Write a prediction now. Reranking after evaluation is another exploratory analysis.

In [ ]:
EVALUATION_COLUMNS = ['rule', 'A', 'B', 'period', 'baskets', 'A_count', 'B_count',
                      'joint_count', 'support', 'confidence', 'B_baseline', 'lift', 'status']

def evaluate_frozen(rule_table, partitions):
    rows = []
    for rule in rule_table.itertuples(index=False):
        for period, table in partitions.items():
            evidence = rule_evidence(table, {rule.A}, {rule.B})
            if evidence.baskets == 0: status = 'empty population'
            elif evidence.A_count == 0: status = 'antecedent absent; confidence undefined'
            elif evidence.B_count == 0: status = 'consequent absent; lift undefined'
            elif evidence.joint_count == 0: status = 'both present; no co-occurrence'
            else: status = 'observed co-occurrence; not a pass label'
            record = evidence.to_dict()
            for field in ['baskets', 'A_count', 'B_count', 'joint_count']:
                record[field] = int(record[field])
            rows.append({'rule': f'{rule.A} → {rule.B}', 'A': rule.A, 'B': rule.B,
                         'period': period, **record, 'status': status})
    return pd.DataFrame(rows, columns=EVALUATION_COLUMNS)

evaluation = evaluate_frozen(frozen_rules, {'discovery': discovery, 'validation': validation})
display(evaluation.round(3))
if evaluation.empty:
    print('No rules selected on discovery. Record that result; do not mine validation for replacements.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    names = [f'{row.A} → {row.B}' for row in frozen_rules.itertuples()]
    x = np.arange(len(names))
    for ax, metric in zip(axes, ['confidence', 'lift']):
        for shift, period, color in [(-.18, 'discovery', '#476f95'), (.18, 'validation', '#d58b47')]:
            values = evaluation[evaluation.period.eq(period)][metric].to_numpy(float)
            ax.bar(x + shift, values, width=.36, label=period, color=color)
            for position, value in zip(x + shift, values):
                if not np.isfinite(value): ax.text(position, 0, 'undefined', rotation=90, va='bottom', fontsize=8)
        ax.set(xticks=x, xticklabels=names, title=metric.capitalize(), ylabel=metric.capitalize())
        ax.tick_params(axis='x', labelsize=9)
    axes[0].set_ylim(0, 1.08)
    axes[1].axhline(1, color='black', linestyle='--', linewidth=1)
    axes[0].legend(fontsize=9)
    fig.suptitle('Exact frozen rules in two periods — consult the counts; no uncertainty intervals')
    plt.tight_layout(); plt.show()

In [ ]:
# Controlled boundary examples: synthetic, not additional retail discoveries.
# Each row below evaluates the SAME specified A → B instead of mining it again.
boundary_rules = pd.DataFrame({'A': ['A'], 'B': ['B']})
boundary_evaluation = evaluate_frozen(boundary_rules, {
    'zero_joint': constructed_baskets(0, 2, 3, 5),
    'A_absent': constructed_baskets(0, 0, 3, 7),
    'B_absent': constructed_baskets(0, 2, 0, 8),
    'empty_population': constructed_baskets(0, 0, 0, 0)})
display(boundary_evaluation.round(3))
print('Zero co-occurrence, undefined confidence, and undefined lift are different outcomes.')

In [ ]:
# TODO 5: Inspect source evidence and the reverse conditional; explain limits.
selected_A, selected_B = frozen_rules.iloc[0] if len(frozen_rules) else (None, None)
if selected_A is not None:
    in_A, in_B = validation[selected_A], validation[selected_B]
    for kind, mask in [('matching', in_A & in_B), ('A without B', in_A & ~in_B)]:
        ids = validation.index[mask]
        print(kind, 'validation invoices:', len(ids))
        if len(ids):
            chosen = ids[0]
            source = validation_lines[validation_lines.InvoiceNo.eq(chosen)]
            print('Inspected invoice:', chosen, '| present selected products:',
                  list(validation.columns[validation.loc[chosen]]))
            display(source[source.StockCode.isin([selected_A, selected_B])]
                [['InvoiceNo', 'InvoiceDate', 'CustomerID', 'StockCode', 'Description', 'Quantity']])
        else:
            print('No such example in this validation period.')
    display(pd.DataFrame({'forward': rule_evidence(validation, {selected_A}, {selected_B}),
                         'reverse': rule_evidence(validation, {selected_B}, {selected_A})}).round(3))
else:
    print('No selected rule to inspect. Explain why the discovery shortlist is empty.')

## Reading C — Persistence is evidence, not permission to stop thinking

**Read and reflect for 10–15 minutes.** Your goal is to distinguish a changed score, a changed question, and a reason to revise a recommendation.

### Small denominators make dramatic fractions easy

A rule with two successes from two antecedent baskets has perfect confidence. A rule with eighty successes from one hundred antecedent baskets has lower confidence but much more information about repeated co-occurrence. Neither is automatically the better action. We need to know the use case, the consequent's prevalence, and how stable each estimate is. With the first rule, one contrary basket would change confidence from 1 to 2/3; with the second, it would change 0.8 to 80/101. The numerical sensitivity comes from the denominator, not from an algorithm becoming unreliable in a mysterious way.

Rare consequents can also produce large lift. If B occurs in very few baskets, dividing by its small baseline can magnify a small joint count. That is why filtering by lift alone is an incomplete evidence policy. A minimum count reduces some extreme low-evidence discoveries, but no count threshold is a certificate of reliability. A rule can meet a threshold because the same enthusiastic customer buys the same pair repeatedly.

### Invoices are not independently sampled people

Suppose a joint count of twenty comes from twenty invoices placed by one buyer. Another rule has twenty joint invoices across eighteen buyers. The invoice-level support calculation is valid for both. The strength of a claim about broad customer behavior is different. Count distinct contributing customers, and inspect whether one history dominates the pattern before using phrases such as “customers tend to.”

Removing the most frequent contributing customer is a useful stress test, but it changes the population. For a fair before/after comparison, remove all that customer's validation invoices, not only matching ones. Otherwise you selectively delete successes while retaining their counterexamples. Even the fair version is descriptive: it asks how dependent the conclusion is on that customer's contribution. It does not prove that customers are independent, identify fraud, or establish performance for unseen buyers.

The alternative route checks evidence volume. Compare an earlier prefix of the validation invoices with the full validation period, using the same frozen rule and catalog. The prefix is contained in the full period; these are overlapping views, not two independent replications. Calendar context and sample volume change together. If a fraction changes, do not attribute the change solely to “more data.” An availability change, seasonal demand, or customer composition could contribute too.

### Thresholds filter discoveries; they do not alter their counts

Raising the discovery joint-count threshold changes which rules qualify for selection. It does not change the underlying evidence for a fixed rule on unchanged baskets. Similarly, sorting by confidence instead of lift changes the shortlist, not the observations. Distinguish this from changing invoice to customer-history baskets, which changes the question, or dropping catalog-empty rows, which changes the reference population. Precise language about the change prevents a selection preference from becoming a false improvement claim.

Our shortlists also remove reverse duplicates, but product-family redundancy remains. Three highly lifted variants of the same design may support one family-level hypothesis rather than three distinct opportunities. Describe their common context. A technically correct table can still overstate the breadth of a finding if every row is treated as a separate discovery.

### There is a cost to looking again

Once you inspect validation and use its results to change a rule, threshold, or catalog, the revised choice is informed by that data. Repeatedly trying alternatives until one looks good consumes the independence of the check. You can explore those alternatives, but label them exploratory. A new final claim needs new evidence, a clearly separated later test, or an evaluation design that accounts for the selection process. Restarting the notebook does not make previously seen observations unseen.

Do not turn today's sensitivity routes into a contest to maximize validation lift. Their purpose is to expose a limitation and qualify the recommendation. A rule that retains lift above one in both periods may deserve further investigation; it still has no guaranteed future usefulness. Likewise, a failed check need not mean a meaningless original pattern: an authentic relationship can depend on a particular season or context. Be specific about the conditions supported by the evidence.

### From evidence to a proportionate next step

A defensible recommendation separates observation, interpretation, and action. Observation: a fixed pair co-occurred at stated rates in both periods. Interpretation: product descriptions suggest a shared purchasing context. Action: inspect availability and customer coverage, or design a controlled recommendation test if the business question is causal. Profit also requires costs, margins, capacity, and a baseline policy; no such calculation follows from lift alone.

### Checkpoint

Why is 2/2 a fragile basis for confidence? Why remove all invoices from a dominant customer in a stress test? Does a validation prefix provide another independent test? What must change in your reporting if you choose a new winner after seeing validation? Write a recommendation that would remain useful even if you advise no action yet.

<details><summary>Self-check</summary>

One additional contrary basket changes 2/2 to 2/3. Remove the entire customer's contribution to define a coherent reduced population. The prefix overlaps the full period and also changes calendar coverage. A revised winner is an exploratory choice informed by validation; it needs a fresh check for an independent performance claim. A useful recommendation can ask for broader customer coverage or stock-availability evidence before any intervention.

</details>

## Investigation 2 — What is your conclusion leaning on?

**40 minutes. Choose one route.** Keep the frozen shortlist, catalog, and invoice definition. Do not use this activity to choose the best-looking replacement rule.

**Route A — `customer_concentration`:** for the first frozen rule, count joint invoices by customer. Report the number of contributing customers and the largest contributor's share of the joint count. Remove all validation invoices of that customer and recalculate the frozen rule on the remaining population. Explain how much the result depends on one history. This is a stress test on a changed population, not evidence about a new customer sample.

**Route B — `evidence_volume`:** evaluate all frozen rules on the first half of validation invoices ordered by date and invoice ID, then on the full validation period. Report date coverage and the actual counts. A large metric change with a small antecedent count deserves inspection. These views overlap and mix volume with calendar context; they do not measure independent replications or prove a sample-size effect.

Use five minutes for a prediction, twenty for the route and source inspection, ten to revise your claim, and five to compare explanations. The default code supplies the data manipulation; focus on explaining what was held fixed and what changed.

Everyone then inspects the discovery threshold table below. Identify a frozen rule that would be excluded by a higher threshold if one exists. Its validation values remain computable even if it would not have been selected. If every selected rule survives the listed thresholds, state that. Do not search for a failure merely to fill a box.

**TODO 6:** Run one route and record the count-level consequence. **TODO 7:** Explain the threshold table without calling a smaller rule list a more accurate model. Name a limitation the route cannot resolve. An absent joint event or empty shortlist has a message and is a result you can explain.

**Done when:** your evidence table contains the frozen claim, the original observation, the controlled challenge, its numerical consequence, and a smaller or better-qualified claim. If a partner chooses the other route, compare what the two checks can and cannot establish.

<details><summary>Hint</summary>

Start with “We kept ... fixed and changed ... .” Then state one count and one denominator before interpreting a score. If the score barely changes, ask whether that addresses your strongest concern or only one narrow concern.

</details>

In [ ]:
# TODO 6: Choose customer_concentration or evidence_volume; never refit the shortlist.
challenge_route = 'customer_concentration'
if challenge_route not in {'customer_concentration', 'evidence_volume'}:
    raise ValueError('Choose customer_concentration or evidence_volume.')

invoice_meta = (validation_lines[['InvoiceNo', 'InvoiceDate', 'CustomerID']]
                .drop_duplicates().set_index('InvoiceNo').reindex(validation.index))
contributions = pd.Series(dtype='int64', name='joint_invoices')
dominant_customer = None
if challenge_route == 'customer_concentration':
    if selected_A is None:
        print('No frozen rule: explain the discovery outcome instead.')
    else:
        joint_mask = validation[selected_A] & validation[selected_B]
        contributions = (invoice_meta.loc[joint_mask, 'CustomerID'].value_counts()
                         .rename_axis('CustomerID').rename('joint_invoices'))
        print('Distinct joint-event customers:', len(contributions))
        display(contributions.head(5))
        if contributions.empty:
            print('No joint event; there is no dominant joint contributor to remove.')
        else:
            # Deterministic tie break by ID; remove ALL their invoices, not only joint ones.
            contribution_table = contributions.reset_index().sort_values(
                ['joint_invoices', 'CustomerID'], ascending=[False, True])
            dominant_customer = contribution_table.iloc[0].CustomerID
            keep = invoice_meta.CustomerID.ne(dominant_customer)
            reduced_validation = validation.loc[keep]
            print('Removed customer:', dominant_customer, '| Share of joint invoices:',
                  round(contribution_table.iloc[0].joint_invoices / contributions.sum(), 3),
                  '| All their validation invoices removed:', int((~keep).sum()))
            challenge_evaluation = evaluate_frozen(frozen_rules.iloc[:1],
                {'all_validation': validation, 'without_dominant_customer': reduced_validation})
            display(challenge_evaluation.round(3))
else:
    ordered_ids = (invoice_meta.reset_index().sort_values(['InvoiceDate', 'InvoiceNo']).InvoiceNo)
    prefix_ids = ordered_ids.iloc[:max(1, len(ordered_ids) // 2)]
    validation_prefix = validation.loc[list(prefix_ids)]
    print('Prefix invoices:', len(validation_prefix), '| Date range:',
          invoice_meta.loc[prefix_ids, 'InvoiceDate'].min(), 'to',
          invoice_meta.loc[prefix_ids, 'InvoiceDate'].max())
    challenge_evaluation = evaluate_frozen(frozen_rules,
        {'validation_prefix': validation_prefix, 'all_validation': validation})
    display(challenge_evaluation.round(3))

In [ ]:
# TODO 7: These discovery filters change eligibility, not fixed-rule evidence.
threshold_tables = {count: mine_pair_rules(discovery, count) for count in [8, 16, 24]}
display(pd.DataFrame([{'discovery_minimum_joint_count': count, 'directed_candidates': len(table)}
                      for count, table in threshold_tables.items()]))
eligibility_rows = []
for rule in frozen_rules.itertuples():
    counts = rule_evidence(discovery, {rule.A}, {rule.B})
    eligibility_rows.append({'rule': f'{rule.A} → {rule.B}', 'discovery_joint_count': counts.joint_count,
        **{f'qualifies_at_{count}': counts.joint_count >= count for count in threshold_tables}})
display(pd.DataFrame(eligibility_rows))
print('Frozen identities unchanged:', freeze_record['rules'] == tuple(map(tuple, frozen_rules.to_numpy())))

## Synthesis — a recommendation that survives your own objections

Use fifteen minutes for a 150–250-word evidence memo. Cover the three frozen rules, the actual source inspection, and your controlled challenge. State the strongest supported observation, one plausible explanation, one important limitation, and a proportionate next step. You do not need three positive results. If fewer rules were discovered, explain the available evidence instead of manufacturing more.

**TODO 8:** Fill an evidence table with `frozen_claim`, `discovery_counts`, `validation_counts`, `baseline_change`, `challenge`, `revised_claim`, and `next_evidence`. Refer to the cutoff, basket unit, and selected catalog. Revisit your opening prediction about perfect discovery confidence. Distinguish “observed again,” “failed to reproduce,” “too little evidence,” and “not defined.”

A useful sentence structure is: “Under our fixed invoice definition, [rule] changed from [joint/antecedent] to [joint/antecedent], while B's prevalence changed from [...] to [...]. The [challenge] showed [...]. This supports [...], but does not establish [...]; before acting we would need [...].” Supply your own values and interpretation. A recommendation to collect availability data or test an intervention is different from claiming the current notebook already established its benefit.

All activities are formative. Use only explicitly released assignment briefs and confirmed deadlines for assessment; this lecture does not release a new submission. Next, text mining asks what happens when items are words or features we must construct from documents. The same discipline returns: define the unit, fit the representation on the appropriate data, compare a baseline, and inspect what the evidence supports.

## Optional extension — many searches in a world with no designed associations

The following bounded simulation generates independent binary products with known probability 0.15 in each basket. The largest observed lift across qualifying pairs can exceed one even though the generator contains no designed associations. We repeat this whole experiment forty times. The histogram describes maxima chosen from many candidates; it is not the distribution for a rule specified in advance.

This is a synthetic illustration, not a null test for the retail shortlist. It does not reproduce invoice sizes, customer dependence, seasonality, or the retail selection process. Do not attach its frequencies to a retail rule as a p-value. Before running, predict what allowing more products could do to the opportunity to find an extreme pair, while keeping the count boundary fixed. Try 10 versus 30 products as an optional second comparison; the default uses 20.

In [ ]:
rng = np.random.default_rng(42)
simulated_products = 20  # keep between 2 and 30
if not 2 <= simulated_products <= 30:
    raise ValueError('Use 2 to 30 products for this bounded illustration.')
maximum_lifts = []
for repetition in range(40):
    simulated = (rng.random((120, simulated_products)) < .15).astype(np.int64)
    marginal = simulated.sum(axis=0)
    joint = simulated.T @ simulated
    lifts = []
    for i in range(simulated_products):
        for j in range(i + 1, simulated_products):
            if joint[i, j] >= 3:
                lifts.append(joint[i, j] * len(simulated) / (marginal[i] * marginal[j]))
    maximum_lifts.append(max(lifts) if lifts else np.nan)
finite = np.array(maximum_lifts)[np.isfinite(maximum_lifts)]
print('Repetitions with at least one qualifying pair:', len(finite), 'of', len(maximum_lifts))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(finite, bins=10, color='#476f95', edgecolor='white')
ax.axvline(1, color='black', linestyle='--', label='Generator independence baseline')
ax.set(xlabel='Largest empirical pair lift (joint count ≥ 3)', ylabel='Simulated repetitions',
       title='Selecting the largest value favors extremes, even with independent products')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()